# Faithful-HRM — GSM8K math reasoning (digit-aware bridge + HRM/ACT)

**What changed vs v3.1**
1. **Faithful dataset.** Traces come from GSM8K's own `<<a op b = c>>` annotations
   (number-grounding 73–75% vs 24% in the LLM-confabulated set). No confabulation.
2. **Digit-aware bridge (architectural).** Operands are encoded as DIGIT sequences,
   not a lossy `log1p` scalar — the HRM can now see digits and learn carry arithmetic.
3. **3-stage training (same distribution):**
   - **Stage 1 – Pretrain** on synthetic arithmetic graphs sampled to match GSM8K's
     op-mix / step-count / magnitude distribution (unlimited data).
   - **Stage 2 – Curriculum finetune** on faithful GSM8K, phased by step count.
   - **Stage 3 – Finetune** on all faithful GSM8K + number augmentation, ACT on.
4. Kept: HRM H/L core, ACT halting, Adam-atan2, deep supervision (aux loss).

**Add Data:** upload `gsm8k_faithful_train.json`, `gsm8k_faithful_val.json`,
`gsm8k_faithful_test.json` (produced by `build_faithful_gsm8k.py`).


In [ ]:
# Cell 1 — Config
import os, glob, torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| torch", torch.__version__)

# Auto-locate the faithful dataset wherever it is mounted (Kaggle attaches it
# under /kaggle/input/<slug>). Falls back to env var, then local ./data.
_hits = glob.glob("/kaggle/input/**/gsm8k_faithful_train.json", recursive=True)
DATA_ROOT = os.path.dirname(_hits[0]) if _hits else os.environ.get("DATA_ROOT", "data")
print("DATA_ROOT:", DATA_ROOT)
assert os.path.exists(f"{DATA_ROOT}/gsm8k_faithful_train.json"), (
    f"faithful files not found under {DATA_ROOT!r}. "
    "Attach the dataset via 'Add Data' and check it contains gsm8k_faithful_*.json.")

CONFIG = dict(
    data_train=f"{DATA_ROOT}/gsm8k_faithful_train.json",
    data_val  =f"{DATA_ROOT}/gsm8k_faithful_val.json",
    data_test =f"{DATA_ROOT}/gsm8k_faithful_test.json",
    savedir   ="/kaggle/working/ckpt_faithful" if os.path.isdir("/kaggle") else "ckpt_faithful",

    # model
    dmodel=256, nheads=8, Hcycles=3, Lcycles=4, Hlayers=4, Llayers=4, max_nodes=40,

    # Encoding: True = HONEST (only leaf numbers fed; model computes+routes every
    # intermediate AND the answer). False = LEAKY ablation (operands+answer fed in,
    # which trivially hits ~100% — use only to demonstrate the leak).
    mask_refs=True,

    # Stage 1 — synthetic pretrain
    pretrain_n=60000, pretrain_epochs=8, pretrain_bs=256, pretrain_lr=6e-4,

    # Stage 2 — curriculum (phase = max step-count; epochs per phase)
    curriculum_phases=[(2, 12), (4, 12), (99, 16)], curr_bs=128, curr_lr=3e-4,

    # Stage 3 — full finetune + augmentation, ACT enabled
    finetune_epochs=80, finetune_bs=128, finetune_lr=2e-4,
    act_max_steps=3, act_min_steps=2, q_loss_weight=0.5, aux_loss_weight=1.0,
    augment_p=0.3, augment_max_value=300,

    # adam-atan2
    optim_betas=(0.9, 0.95), optim_wd=0.01, optim_a=1.27, optim_b=1.0,
)
os.makedirs(CONFIG["savedir"], exist_ok=True)
for k, v in CONFIG.items(): print(f"  {k:18s}= {v}")


## Cell 2 — Core: encoding, faithful↔synthetic data, digit-aware HRM (inlined `faithful_hrm.py`)

In [ ]:
# Cell 2 — Core (inlined)
"""
Faithful-HRM core: digit-aware Graph bridge + HRM(H/L)+ACT + digit head,
plus a synthetic arithmetic-graph generator and a 3-stage training driver
(pretrain on synthetic -> curriculum finetune -> full finetune).

This module is the single source of truth; the training notebook inlines it.
Designed to run on CPU (tiny) for smoke tests and CUDA (full) on Kaggle.
"""
import math, json, random, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ───────────────────────── vocab / encoding ─────────────────────────
OP_VOCAB = {"PAD": 0, "add": 1, "sub": 2, "mul": 3, "div": 4, "final_answer": 5}
OP_VOCAB_SIZE = len(OP_VOCAB)
DIGIT_VOCAB = {"PAD": 0, "0": 1, "1": 2, "2": 3, "3": 4, "4": 5, "5": 6,
               "6": 7, "7": 8, "8": 9, "9": 10, "NEG": 11, "EOS": 12}
DIGIT_VOCAB_SIZE = 13
MAX_DIGITS = 8
IDX2DIG = {v: k for k, v in DIGIT_VOCAB.items()}
MAX_ABS = 10 ** MAX_DIGITS - 1


def encode_number(val):
    n = int(round(val))
    d = []
    if n < 0:
        d.append(DIGIT_VOCAB["NEG"]); n = abs(n)
    for ch in str(n):
        d.append(DIGIT_VOCAB[ch])
    d.append(DIGIT_VOCAB["EOS"])
    while len(d) < MAX_DIGITS:
        d.append(DIGIT_VOCAB["PAD"])
    return d[:MAX_DIGITS]


def decode_digits(tokens):
    neg, s, seen = False, "", False
    for t in tokens:
        lbl = IDX2DIG.get(int(t), "PAD")
        if lbl == "PAD":
            continue
        if lbl == "EOS":
            break
        if lbl == "NEG":
            neg, seen = True, True
        else:
            s += lbl; seen = True
    if not seen or not s:
        return -1
    return -int(s) if neg else int(s)


# ───────────────────────── JSON trace -> tensors ─────────────────────────
def parse_graph(trace, max_nodes=40, mask_refs=True):
    """Digit-aware encoding of one trace.

    mask_refs=True (default, the honest setting): an operand that REFERENCES a
    prior node's result is NOT given its value — it is masked to zero, and the
    model must compute that value itself and route it through the graph edges.
    Only leaf literals (the problem's given numbers) are fed. The final_answer
    node is a pure readout (no operands) so the answer is never an input.

    mask_refs=False reproduces the leaky setting (operands + answer fed in) for
    the ablation that demonstrates why it hits ~100%.
    """
    op_ids, opnd_digits, is_ref, dig_tgts = [], [], [], []
    v2i, v2v = {}, {}
    adj = np.zeros((max_nodes, max_nodes), dtype=np.float32)

    def resolve(a):
        if isinstance(a, (int, float)):
            return float(a), False
        if isinstance(a, str) and a in v2v:
            return v2v[a], True
        try:
            return float(a), False
        except Exception:
            return 0.0, False

    for i, step in enumerate(trace.get("steps", [])):
        if i >= max_nodes - 1:
            break
        op = step.get("op", "PAD")
        op_ids.append(OP_VOCAB.get(op, 0))
        v1, r1 = resolve(step.get("arg1", 0))
        v2, r2 = resolve(step.get("arg2", 0))
        if op == "add":   rv = v1 + v2
        elif op == "sub": rv = v1 - v2
        elif op == "mul": rv = v1 * v2
        elif op == "div" and v2 != 0: rv = v1 / v2
        else: rv = v1
        # mask referenced operands: the model must derive them via the graph
        d1 = encode_number(0) if (mask_refs and r1) else encode_number(v1)
        d2 = encode_number(0) if (mask_refs and r2) else encode_number(v2)
        opnd_digits.append([d1, d2])
        is_ref.append([float(r1), float(r2)])
        dig_tgts.append(encode_number(rv))
        rk = step.get("result", "")
        if rk:
            v2i[rk] = i; v2v[rk] = rv
        for k in ("arg1", "arg2"):
            a = step.get(k, "")
            if isinstance(a, str) and a in v2i:
                adj[v2i[a], i] = 1.0
        adj[i, i] = 1.0

    nr = len(op_ids)
    # The answer is read at the node that COMPUTES it (the final step), not a
    # bolt-on readout node — that copy-via-edge indirection bottlenecked even
    # 1-step problems. answer_idx is that node's index.
    fa = trace.get("final_answer", "")
    answer_idx = v2i.get(fa, max(0, nr - 1))

    while len(op_ids) < max_nodes:
        op_ids.append(0)
        opnd_digits.append([[0] * MAX_DIGITS, [0] * MAX_DIGITS])
        is_ref.append([0.0, 0.0])
        dig_tgts.append([0] * MAX_DIGITS)
    return op_ids, opnd_digits, is_ref, adj, dig_tgts, nr, answer_idx


def sample_to_tensors(trace, target, max_nodes, mask_refs=True):
    op_ids, opnd_digits, is_ref, adj, dig_tgts, nr, ans_idx = parse_graph(trace, max_nodes, mask_refs)
    tgt = int(round(float(target)))
    tgt = max(-MAX_ABS, min(MAX_ABS, tgt))
    return {
        "op_ids": torch.tensor(op_ids, dtype=torch.long),
        "opnd_digits": torch.tensor(opnd_digits, dtype=torch.long),   # (N,2,MAX_DIGITS)
        "is_ref": torch.tensor(is_ref, dtype=torch.float32),          # (N,2)
        "adj": torch.tensor(adj, dtype=torch.float32),                # (N,N)
        "node_digit_tgts": torch.tensor(dig_tgts, dtype=torch.long),  # (N,MAX_DIGITS)
        "final_digit_tgt": torch.tensor(encode_number(tgt), dtype=torch.long),
        "raw_target": torch.tensor(tgt, dtype=torch.long),
        "num_real_nodes": torch.tensor(nr, dtype=torch.long),
        "answer_idx": torch.tensor(ans_idx, dtype=torch.long),
    }


def collate(batch):
    return {k: torch.stack([b[k] for b in batch]) for k in batch[0]}


# ───────────────────────── synthetic generator ─────────────────────────
def _reexec_and_label(trace):
    """Compute final answer for a generated trace; None if degenerate."""
    vv = {}
    for s in trace["steps"]:
        v1 = vv[s["arg1"]] if isinstance(s["arg1"], str) else float(s["arg1"])
        v2 = vv[s["arg2"]] if isinstance(s["arg2"], str) else float(s["arg2"])
        op = s["op"]
        if op == "add": rv = v1 + v2
        elif op == "sub": rv = v1 - v2
        elif op == "mul": rv = v1 * v2
        elif op == "div": rv = v1 / v2 if v2 != 0 else None
        else: rv = v1
        if rv is None or not np.isfinite(rv) or abs(rv) > MAX_ABS or rv != int(rv):
            return None
        vv[s["result"]] = rv
    return vv[trace["final_answer"]]


# op mix + step-count distribution approximating faithful GSM8K
_STEP_W = {1: 0.03, 2: 0.24, 3: 0.25, 4: 0.18, 5: 0.12, 6: 0.08, 7: 0.05, 8: 0.03, 9: 0.02}
_OPS = ["mul", "add", "sub", "div"]
_OP_W = [0.37, 0.29, 0.18, 0.16]


def gen_one(rng, max_steps=9):
    steps_choices = [k for k in _STEP_W if k <= max_steps]
    w = np.array([_STEP_W[k] for k in steps_choices]); w = w / w.sum()
    n_steps = int(rng.choice(steps_choices, p=w))
    pool = []                      # (token, value); token is int literal or "vN"
    n_leaf = rng.integers(2, 4)
    for _ in range(n_leaf):
        pool.append((int(rng.integers(1, 100)), None))
    steps = []
    for i in range(n_steps):
        for _try in range(8):
            op = str(rng.choice(_OPS, p=_OP_W))
            a, b = rng.integers(0, len(pool), size=2)
            ta, va = pool[a]; tb, vb = pool[b]
            av = va if va is not None else ta
            bv = vb if vb is not None else tb
            if op == "div":
                if bv == 0 or av % bv != 0:
                    continue
                rv = av // bv
            elif op == "add": rv = av + bv
            elif op == "sub": rv = av - bv
            else: rv = av * bv
            if abs(rv) > MAX_ABS:
                continue
            rk = f"v{i+1}"
            steps.append({"op": op, "arg1": ta, "arg2": tb, "result": rk})
            pool.append((rk, float(rv)))
            break
        else:
            return None
    if not steps:
        return None
    trace = {"steps": steps, "final_answer": steps[-1]["result"]}
    fa = _reexec_and_label(trace)
    if fa is None:
        return None
    return {"trace": trace, "target": float(fa), "question": ""}


def make_synthetic(n, seed=0, max_steps=9):
    rng = np.random.default_rng(seed)
    out = []
    while len(out) < n:
        r = gen_one(rng, max_steps=max_steps)
        if r is not None:
            out.append(r)
    return out


# ───────────────────────── dataset ─────────────────────────
def perturb_constants(trace, max_value=200, rng=None):
    rng = rng or random
    nt = copy.deepcopy(trace); vv = {}
    for s in nt["steps"]:
        for k in ("arg1", "arg2"):
            a = s[k]
            if isinstance(a, (int, float)) or (isinstance(a, str) and a not in vv and _isnum(a)):
                s[k] = float(rng.randint(1, max_value))
        v1 = vv[s["arg1"]] if isinstance(s["arg1"], str) and s["arg1"] in vv else float(s["arg1"])
        v2 = vv[s["arg2"]] if isinstance(s["arg2"], str) and s["arg2"] in vv else float(s["arg2"])
        op = s["op"]
        if op == "add": rv = v1 + v2
        elif op == "sub": rv = v1 - v2
        elif op == "mul": rv = v1 * v2
        elif op == "div": rv = v1 / v2 if v2 != 0 else None
        else: rv = v1
        if rv is None or not np.isfinite(rv) or abs(rv) > MAX_ABS:
            return None, None
        vv[s["result"]] = rv
    fa = vv.get(nt["final_answer"])
    if fa is None or abs(fa) > MAX_ABS:
        return None, None
    return nt, int(round(fa))


def _isnum(a):
    try: float(a); return True
    except Exception: return False


class GraphDataset(Dataset):
    """Holds faithful or synthetic records. Supports step-count curriculum via
    set_phase(max_steps) and optional number augmentation."""
    def __init__(self, records, max_nodes=40, augment=False, augment_p=0.3,
                 augment_max_value=200, mask_refs=True):
        self.max_nodes = max_nodes
        self.augment = augment
        self.augment_p = augment_p
        self.augment_max_value = augment_max_value
        self.mask_refs = mask_refs
        self.records = []
        for it in records:
            tr = it.get("trace", {})
            tgt = float(it.get("target", 0.0))
            if abs(tgt) > MAX_ABS:
                continue
            fa = tr.get("final_answer", "")
            if fa not in {s.get("result", "") for s in tr.get("steps", [])}:
                continue
            self.records.append((tr, tgt, len(tr.get("steps", []))))
        self.active = list(range(len(self.records)))

    def set_phase(self, max_steps):
        self.active = [i for i, (_, _, sc) in enumerate(self.records) if sc <= max_steps]
        return len(self.active)

    def __len__(self):
        return len(self.active)

    def __getitem__(self, idx):
        tr, tgt, _ = self.records[self.active[idx]]
        if self.augment and random.random() < self.augment_p:
            nt, ntgt = perturb_constants(tr, self.augment_max_value)
            if nt is not None:
                return sample_to_tensors(nt, ntgt, self.max_nodes, self.mask_refs)
        return sample_to_tensors(tr, tgt, self.max_nodes, self.mask_refs)


# ───────────────────────── model ─────────────────────────
class DenseGAT(nn.Module):
    def __init__(self, in_f, out_f, heads=4, concat=True, drop=0.1):
        super().__init__()
        self.heads, self.out_f, self.concat = heads, out_f, concat
        self.W = nn.Linear(in_f, heads * out_f, bias=False)
        self.as_ = nn.Linear(out_f, 1, bias=False)
        self.ad = nn.Linear(out_f, 1, bias=False)
        self.drop = nn.Dropout(drop)

    def forward(self, x, adj):
        B, N, _ = x.shape
        xp = self.W(x).reshape(B, N, self.heads, self.out_f)
        s = self.as_(xp).squeeze(-1); d = self.ad(xp).squeeze(-1)
        e = F.leaky_relu(s.unsqueeze(2) + d.unsqueeze(1), 0.2)
        e = e.masked_fill(adj.unsqueeze(-1) == 0, -1e4)
        attn = self.drop(F.softmax(e, dim=2))
        h = torch.einsum("bnjh,bjhd->bnhd", attn, xp)
        return h.reshape(B, N, self.heads * self.out_f) if self.concat else h.mean(2)


class DigitAwareBridge(nn.Module):
    """Encodes each node from op + DIGIT-level operands + is_ref, then GAT."""
    def __init__(self, d, d_op=64, d_dig=32, gh=128, gl=3, heads=4):
        super().__init__()
        self.op_emb = nn.Embedding(OP_VOCAB_SIZE, d_op)
        self.dig_emb = nn.Embedding(DIGIT_VOCAB_SIZE, d_dig)
        self.opnd_proj = nn.Linear(MAX_DIGITS * d_dig, d)        # per operand
        self.fuse = nn.Linear(d_op + 2 * d + 2, d)
        self.gats = nn.ModuleList()
        ind, self.is_last = d, []
        for i in range(gl):
            out, co = (d, False) if i == gl - 1 else (gh, True)
            self.gats.append(DenseGAT(ind, out, heads=heads, concat=co))
            self.is_last.append(i == gl - 1)
            ind = out * (heads if co else 1)

    def forward(self, op_ids, opnd_digits, is_ref, adj):
        B, N, _, _ = opnd_digits.shape
        oe = self.op_emb(op_ids)                                  # (B,N,d_op)
        de = self.dig_emb(opnd_digits)                            # (B,N,2,MD,d_dig)
        de = de.reshape(B, N, 2, -1)                              # (B,N,2,MD*d_dig)
        opr = self.opnd_proj(de)                                 # (B,N,2,d)
        x = torch.cat([oe, opr[:, :, 0], opr[:, :, 1], is_ref], dim=-1)
        x = self.fuse(x)
        pad = (op_ids == 0)
        for layer, last in zip(self.gats, self.is_last):
            pm = pad.unsqueeze(2) | pad.unsqueeze(1)
            a2 = adj.clone(); a2[pm] = 0.0
            x = layer(x, a2)
            if not last:
                x = F.elu(x)
        return x


def rms_norm(x, eps=1e-5):
    return x * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + eps).to(x.dtype)


class SwiGLU(nn.Module):
    def __init__(self, d, ex=2.0):
        super().__init__()
        i = int(round(ex * d * 2 / 3)); i = (i + 127) // 128 * 128
        self.gu = nn.Linear(d, i * 2, bias=False); self.dn = nn.Linear(i, d, bias=False)

    def forward(self, x):
        g, u = self.gu(x).chunk(2, dim=-1)
        return self.dn(F.silu(g) * u)


class Block(nn.Module):
    def __init__(self, d, h, ex=2.0):
        super().__init__()
        self.h, self.hd = h, d // h
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.op = nn.Linear(d, d, bias=False); self.mlp = SwiGLU(d, ex)

    def forward(self, x, mask=None):
        B, N, D = x.shape
        q, k, v = self.qkv(x).reshape(B, N, 3, self.h, self.hd).unbind(2)
        q, k, v = (t.transpose(1, 2) for t in (q, k, v))
        s = (q @ k.transpose(-2, -1)) / math.sqrt(self.hd)
        if mask is not None: s = s + mask
        o = (F.softmax(s, -1) @ v).transpose(1, 2).reshape(B, N, D)
        x = rms_norm(x + self.op(o)); x = rms_norm(x + self.mlp(x))
        return x


class Module_(nn.Module):
    def __init__(self, nl, d, h, ex=2.0):
        super().__init__()
        self.layers = nn.ModuleList([Block(d, h, ex) for _ in range(nl)])

    def forward(self, hid, inj, mask=None):
        hid = hid + inj
        for l in self.layers: hid = l(hid, mask)
        return hid


class FaithfulHRM(nn.Module):
    def __init__(self, d=256, heads=8, Hc=3, Lc=4, Hl=4, Ll=4, ex=2.0, slen=40):
        super().__init__()
        self.Hc, self.Lc = Hc, Lc
        self.bridge = DigitAwareBridge(d)
        self.pos = nn.Embedding(slen, d)
        self.Hmod = Module_(Hl, d, heads, ex); self.Lmod = Module_(Ll, d, heads, ex)
        self.Hi = nn.Parameter(torch.randn(d) * 0.02)
        self.Li = nn.Parameter(torch.randn(d) * 0.02)
        self.dhead = nn.Linear(d, MAX_DIGITS * DIGIT_VOCAB_SIZE)
        self.qnorm = nn.LayerNorm(d)
        self.qhead = nn.Linear(d, 2)
        nn.init.zeros_(self.qhead.weight); self.qhead.bias.data.copy_(torch.tensor([-5., -5.]))

    def encode(self, b):
        x = self.bridge(b["op_ids"], b["opnd_digits"], b["is_ref"], b["adj"])
        N = b["op_ids"].shape[1]
        x = x + self.pos(torch.arange(N, device=x.device).unsqueeze(0))
        pad = (b["op_ids"] == 0)
        amask = pad.float().unsqueeze(1).unsqueeze(1) * -1e4
        return x, amask

    def init_carry(self, B, N, device):
        zH = self.Hi.view(1, 1, -1).expand(B, N, -1).contiguous()
        zL = self.Li.view(1, 1, -1).expand(B, N, -1).contiguous()
        return zH, zL

    def step(self, b, zH, zL):
        x, amask = self.encode(b)
        B, N, _ = x.shape
        with torch.no_grad():
            for h in range(self.Hc):
                for l in range(self.Lc):
                    if h == self.Hc - 1 and l == self.Lc - 1:
                        continue
                    zL = self.Lmod(zL, zH + x, amask)
                if h != self.Hc - 1:
                    zH = self.Hmod(zH, zL, amask)
        zL = self.Lmod(zL, zH + x, amask)
        zH = self.Hmod(zH, zL, amask)
        dl = self.dhead(zH).reshape(B, N, MAX_DIGITS, DIGIT_VOCAB_SIZE)
        ql = self.qhead(self.qnorm(zH[:, 0]))
        return dl, ql[:, 0], ql[:, 1], zH.detach(), zL.detach()

    def forward(self, b, max_steps=1):
        x, _ = self.encode(b)
        B, N, _ = x.shape
        zH, zL = self.init_carry(B, N, b["op_ids"].device)
        for _ in range(max_steps):
            dl, qh, qc, zH, zL = self.step(b, zH, zL)
        return dl, qh, qc


# ───────────────────────── loss / eval ─────────────────────────
def segment_loss(dl, final_tgt, node_tgts, num_real, qh, qc, next_q, aux_w, q_w, answer_idx):
    B, N, MD, V = dl.shape
    # main loss on the node that COMPUTES the answer
    idx = answer_idx.clamp(min=0)
    final_logits = dl[torch.arange(B), idx]                       # (B,MD,V)
    main = F.cross_entropy(final_logits.reshape(-1, V), final_tgt.reshape(-1),
                           ignore_index=DIGIT_VOCAB["PAD"])
    # aux loss on all real nodes
    node_mask = (torch.arange(N, device=dl.device).unsqueeze(0) < num_real.unsqueeze(1))
    aux_logits = dl.reshape(B * N, MD, V)
    aux_tgt = node_tgts.reshape(B * N, MD)
    aux_ce = F.cross_entropy(aux_logits.reshape(-1, V), aux_tgt.reshape(-1),
                             ignore_index=DIGIT_VOCAB["PAD"], reduction="none")
    aux_ce = aux_ce.reshape(B * N, MD).mean(1).reshape(B, N)
    aux = (aux_ce * node_mask).sum() / node_mask.sum().clamp(min=1)
    # ACT q-loss
    q = F.binary_cross_entropy_with_logits(qh, next_q) + \
        F.binary_cross_entropy_with_logits(qc, next_q)
    return main + aux_w * aux + q_w * q, main


@torch.no_grad()
def evaluate(model, loader, device, act_steps=1):
    model.eval()
    exact = total = digit_ok = digit_tot = no_out = 0
    for b in loader:
        b = {k: v.to(device) for k, v in b.items()}
        dl, _, _ = model(b, max_steps=act_steps)
        idx = b["answer_idx"].clamp(min=0)
        pred = dl[torch.arange(dl.shape[0]), idx].argmax(-1)        # (B,MD)
        for i in range(pred.shape[0]):
            p = decode_digits(pred[i].tolist())
            t = int(b["raw_target"][i])
            if p == -1: no_out += 1
            exact += int(p == t); total += 1
        gt = b["final_digit_tgt"]
        m = gt != DIGIT_VOCAB["PAD"]
        digit_ok += ((pred == gt) & m).sum().item(); digit_tot += m.sum().item()
    return {"exact": exact / max(total, 1), "digit": digit_ok / max(digit_tot, 1),
            "no_out": no_out, "n": total}


In [ ]:
# Cell 3 — Adam-atan2 optimizer + reusable training-stage driver
import math, time, json
from torch.optim.optimizer import Optimizer
from torch.utils.data import DataLoader

class AdamATan2(Optimizer):
    """Adam with bounded atan2 update + decoupled weight decay (HRM paper §3.6)."""
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.95), weight_decay=0.0, a=1.27, b=1.0):
        super().__init__(params, dict(lr=lr, betas=betas, weight_decay=weight_decay, a=a, b=b))
    @torch.no_grad()
    def step(self, closure=None):
        for g in self.param_groups:
            b1, b2 = g["betas"]
            for p in g["params"]:
                if p.grad is None: continue
                st = self.state[p]
                if not st:
                    st["step"]=0; st["m"]=torch.zeros_like(p); st["v"]=torch.zeros_like(p)
                m, v = st["m"], st["v"]; st["step"] += 1; t = st["step"]
                if g["weight_decay"]: p.mul_(1 - g["lr"]*g["weight_decay"])
                m.mul_(b1).add_(p.grad, alpha=1-b1)
                v.mul_(b2).addcmul_(p.grad, p.grad, value=1-b2)
                mh = m/(1-b1**t); vh = v/(1-b2**t)
                p.add_(torch.atan2(mh, vh.sqrt()*g["b"]), alpha=-g["lr"]*g["a"])

def make_opt(model):
    return AdamATan2(model.parameters(), lr=CONFIG["finetune_lr"],
                     betas=CONFIG["optim_betas"], weight_decay=CONFIG["optim_wd"],
                     a=CONFIG["optim_a"], b=CONFIG["optim_b"])

def run_stage(model, train_loader, val_loader, *, epochs, lr, tag,
              act_steps=1, act_min=1, aux_w=1.0, q_w=0.0, eval_every=2):
    """One training stage. act_steps>1 enables multi-segment ACT with halt supervision."""
    opt = make_opt(model)
    for pg in opt.param_groups: pg["lr"] = lr
    total = max(1, epochs*len(train_loader)); warm = max(1, int(0.05*total)); step = 0
    best = 0.0
    print(f"\n===== STAGE: {tag}  (epochs={epochs}, lr={lr}, act_steps={act_steps}) =====")
    for ep in range(epochs):
        model.train(); el = es = 0
        for b in train_loader:
            b = {k: v.to(DEVICE) for k, v in b.items()}
            lr_now = lr*step/warm if step < warm else lr*(0.05 + 0.95*0.5*(1+math.cos(
                math.pi*min(1.0,(step-warm)/max(1,total-warm)))))
            for pg in opt.param_groups: pg["lr"] = lr_now
            B, N = b["op_ids"].shape
            zH, zL = model.init_carry(B, N, DEVICE)
            segs = []
            for s in range(act_steps):
                dl, qh, qc, zH, zL = model.step(b, zH, zL); segs.append((dl, qh, qc))
            loss = 0.0
            for s, (dl, qh, qc) in enumerate(segs):
                nq = (torch.sigmoid(torch.maximum(segs[s+1][1], segs[s+1][2])).detach()
                      if s+1 < len(segs) else torch.sigmoid(qh).detach())
                w = 0.0 if (s+1) < act_min else q_w
                sl, _ = segment_loss(dl, b["final_digit_tgt"], b["node_digit_tgts"],
                                     b["num_real_nodes"], qh, qc, nq, aux_w, w, b["answer_idx"])
                loss = loss + sl
            loss = loss/len(segs)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); step += 1; el += loss.item(); es += 1
        if (ep+1) % eval_every == 0 or ep == epochs-1:
            m = evaluate(model, val_loader, DEVICE, act_steps=act_steps)
            flag = ""
            if m["exact"] > best:
                best = m["exact"]; flag = " *"
                torch.save(model.state_dict(), os.path.join(CONFIG["savedir"], f"best_{tag}.pt"))
            print(f"  ep{ep+1:>3} loss={el/es:6.3f}  val_exact={m['exact']*100:5.2f}%  "
                  f"val_digit={m['digit']*100:5.1f}%  no_out={m['no_out']}{flag}")
    print(f"  -> best {tag} val_exact = {best*100:.2f}%")
    return best


In [ ]:
# Cell 4 — Data: load faithful splits, generate same-distribution synthetic
import json
faithful_train = json.load(open(CONFIG["data_train"]))
faithful_val   = json.load(open(CONFIG["data_val"]))
faithful_test  = json.load(open(CONFIG["data_test"]))
print(f"faithful  train={len(faithful_train)}  val={len(faithful_val)}  test={len(faithful_test)}")

print(f"generating {CONFIG['pretrain_n']} synthetic arithmetic graphs (same distribution)...")
t0 = time.time()
synth = make_synthetic(CONFIG["pretrain_n"], seed=0, max_steps=9)
print(f"  done in {time.time()-t0:.1f}s")

MN = CONFIG["max_nodes"]; MR = CONFIG["mask_refs"]
print("encoding:", "HONEST (mask_refs=True)" if MR else "LEAKY ablation (mask_refs=False)")
pre_ds   = GraphDataset(synth, max_nodes=MN, mask_refs=MR)
train_ds = GraphDataset(faithful_train, max_nodes=MN, mask_refs=MR,
                        augment=True, augment_p=CONFIG["augment_p"],
                        augment_max_value=CONFIG["augment_max_value"])
val_ds   = GraphDataset(faithful_val, max_nodes=MN, mask_refs=MR)
test_ds  = GraphDataset(faithful_test, max_nodes=MN, mask_refs=MR)
val_loader  = DataLoader(val_ds,  batch_size=256, collate_fn=collate)
test_loader = DataLoader(test_ds, batch_size=256, collate_fn=collate)

model = FaithfulHRM(d=CONFIG["dmodel"], heads=CONFIG["nheads"],
                    Hc=CONFIG["Hcycles"], Lc=CONFIG["Lcycles"],
                    Hl=CONFIG["Hlayers"], Ll=CONFIG["Llayers"], slen=MN).to(DEVICE)
print(f"FaithfulHRM params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")


## Stage 1 — Pretrain on synthetic arithmetic graphs

In [ ]:
# Cell 5 — Stage 1: pretrain (teaches exact multi-step arithmetic, infinite data)
pre_loader = DataLoader(pre_ds, batch_size=CONFIG["pretrain_bs"], shuffle=True,
                        collate_fn=collate, drop_last=True)
run_stage(model, pre_loader, val_loader,
          epochs=CONFIG["pretrain_epochs"], lr=CONFIG["pretrain_lr"],
          tag="pretrain", act_steps=1, aux_w=1.0, q_w=0.0)


## Stage 2 — Curriculum finetune on faithful GSM8K (easy→hard by step count)

In [ ]:
# Cell 6 — Stage 2: curriculum by step count
for max_steps, epochs in CONFIG["curriculum_phases"]:
    n = train_ds.set_phase(max_steps)
    loader = DataLoader(train_ds, batch_size=CONFIG["curr_bs"], shuffle=True,
                        collate_fn=collate, drop_last=False)
    run_stage(model, loader, val_loader, epochs=epochs, lr=CONFIG["curr_lr"],
              tag=f"curr<=s{max_steps}", act_steps=1, aux_w=CONFIG["aux_loss_weight"], q_w=0.0)
train_ds.set_phase(99)


## Stage 3 — Full finetune + augmentation, ACT enabled

In [ ]:
# Cell 7 — Stage 3: full finetune with multi-segment ACT + number augmentation
train_ds.set_phase(99)
loader = DataLoader(train_ds, batch_size=CONFIG["finetune_bs"], shuffle=True,
                    collate_fn=collate, drop_last=False)
run_stage(model, loader, val_loader, epochs=CONFIG["finetune_epochs"],
          lr=CONFIG["finetune_lr"], tag="finetune",
          act_steps=CONFIG["act_max_steps"], act_min=CONFIG["act_min_steps"],
          aux_w=CONFIG["aux_loss_weight"], q_w=CONFIG["q_loss_weight"])


## Test-set evaluation

In [ ]:
# Cell 8 — Final held-out GSM8K test evaluation
best = os.path.join(CONFIG["savedir"], "best_finetune.pt")
if os.path.exists(best):
    model.load_state_dict(torch.load(best, map_location=DEVICE))
m = evaluate(model, test_loader, DEVICE, act_steps=CONFIG["act_max_steps"])
print(f"TEST  exact-match={m['exact']*100:.2f}%  digit-acc={m['digit']*100:.1f}%  "
      f"no_output={m['no_out']}/{m['n']}")


In [ ]:
# Cell 9 — Per-step-count accuracy breakdown (shows where reasoning holds/breaks)
from collections import defaultdict
import torch
buckets = defaultdict(lambda: [0, 0])
model.eval()
with torch.no_grad():
    for tr, tgt, sc in [(r[0], r[1], r[2]) for r in test_ds.records]:
        b = collate([sample_to_tensors(tr, tgt, CONFIG["max_nodes"], CONFIG["mask_refs"])])
        b = {k: v.to(DEVICE) for k, v in b.items()}
        dl, _, _ = model(b, max_steps=CONFIG["act_max_steps"])
        idx = int(b["answer_idx"][0])
        p = decode_digits(dl[0, idx].argmax(-1).tolist())
        buckets[sc][1] += 1; buckets[sc][0] += int(p == int(round(tgt)))
print("steps  acc      n")
for sc in sorted(buckets):
    ok, n = buckets[sc]
    print(f"  {sc:>2}  {ok/n*100:5.1f}%  {n:>4}")
